# Bronze layer
Load the raw OpenAlex records (JSONL) as they are into a Delta table.

In [0]:
# Volume holding the JSONL files downloaded from the OpenAlex API
RAW_PATH = "/Volumes/workspace/openalex/raw"

# List the files in the volume
display(dbutils.fs.ls(RAW_PATH))

In [0]:
# Read every JSONL file in the folder (one JSON record per line).
# Spark scans the data to infer the schema, including nested structs.
df_raw = spark.read.json(RAW_PATH)

df_raw.printSchema()

In [0]:
# count() is an action: it forces Spark to actually read the data
df_raw.count()

In [0]:
from pyspark.sql import functions as F

# Keep the source file name: the university is only recorded there.
# Add the load time so every row can be traced back to an ingestion run.
df_bronze = (
    spark.read.json(RAW_PATH)
    .select(
        "*",
        F.col("_metadata.file_name").alias("_source_file"),
        F.current_timestamp().alias("_ingested_at"),
    )
)

display(df_bronze.groupBy("_source_file").count())

In [0]:
# Overwrite so the notebook can be re-run from scratch at any time
(
    df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.openalex.bronze_works")
)

In [0]:
%sql
SELECT _source_file, COUNT(*) AS n_records
FROM workspace.openalex.bronze_works
GROUP BY _source_file
ORDER BY _source_file

In [0]:
%sql
DESCRIBE HISTORY workspace.openalex.bronze_works

In [0]:
%sql
COMMENT ON TABLE workspace.openalex.bronze_works IS
  'Raw OpenAlex works (articles, reviews, conference papers, 2015-2025) of UDC, USC and UVIGO, loaded as-is from the API exports. One row per record and university file: works shared by several universities appear more than once.';